# 時間定義修正後の30分・60分比較

**Historical archive / 過去の研究記録**

原本のコードを保持しています。独立実行や現在の検証基準への適合は保証しません。前のセルの変数に依存する箇所があります。実行入口は `../08_trade_quality.ipynb` を参照してください。

保存出力は `../../results/legacy/`、既知の問題は `../../docs/AUDIT.md` に整理しています。


## 元Notebookのセル 14

出典: `FX.ipynb`、0始まりのindex=13。コード内容は変更していません。

In [ ]:
# ============================================================
# FX機械学習 完全修正版 前半
#
# ・時間定義を統一
# ・30分 / 60分モデル対応
# ・時間減衰ウェイト
# ・MOVE → Direction 二段階AI
# ・Nested Walk-Forward
# ・TP / SLを学習側だけで選択
# ============================================================


# ============================================================
# 1. ライブラリ
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier


# ============================================================
# 2. 基本設定
# ============================================================

SYMBOL = "JPY=X"

PERIOD = "60d"

INTERVAL = "5m"


# MOVE判定
# 0.05%以上の値動き
MOVE_THRESHOLD = 0.0005


# 取引コスト仮定
TRADING_COST = 0.0000133


# 古いデータをどれくらい弱くするか
HALF_LIFE_DAYS = 20


# Walk-Forward分割数
N_SPLITS = 5


# 初期資金
INITIAL_CAPITAL = 10000


# レバレッジ比較
LEVERAGES = [
    1,
    3,
    5
]


# MOVE確率候補
MOVE_PROB_LIST = [
    0.55,
    0.60,
    0.65,
    0.70
]


# Direction確率候補
DIRECTION_PROB_LIST = [
    0.55,
    0.60,
    0.65,
    0.70
]


# TP候補
TP_LIST = [
    0.0005,   # 0.05%
    0.0008,   # 0.08%
    0.0010    # 0.10%
]


# SL候補
SL_LIST = [
    0.0005,   # 0.05%
    0.0007,   # 0.07%
    0.0010    # 0.10%
]


# ============================================================
# 3. USD/JPY 5分足取得
# ============================================================

df = yf.download(
    SYMBOL,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=False,
    progress=False
)


# yfinanceの列名が2段なら1段へ
if df.columns.nlevels > 1:

    df.columns = (
        df.columns
        .get_level_values(0)
    )


# 日本時間
if df.index.tz is not None:

    df.index = (
        df.index
        .tz_convert(
            "Asia/Tokyo"
        )
    )


print(
    "取得した5分足:",
    len(df)
)


# ============================================================
# 4. リターン特徴量
# ============================================================

df["return_5m"] = (
    df["Close"]
    .pct_change(1)
)

df["return_15m"] = (
    df["Close"]
    .pct_change(3)
)

df["return_30m"] = (
    df["Close"]
    .pct_change(6)
)

df["return_1h"] = (
    df["Close"]
    .pct_change(12)
)

df["return_2h"] = (
    df["Close"]
    .pct_change(24)
)


# ============================================================
# 5. 移動平均
# ============================================================

df["MA5"] = (
    df["Close"]
    .rolling(5)
    .mean()
)

df["MA20"] = (
    df["Close"]
    .rolling(20)
    .mean()
)

df["MA50"] = (
    df["Close"]
    .rolling(50)
    .mean()
)


df["MA5_distance"] = (
    df["Close"]
    / df["MA5"]
    - 1
)

df["MA20_distance"] = (
    df["Close"]
    / df["MA20"]
    - 1
)

df["MA50_distance"] = (
    df["Close"]
    / df["MA50"]
    - 1
)


df["MA5_slope"] = (
    df["MA5"]
    .pct_change(3)
)

df["MA20_slope"] = (
    df["MA20"]
    .pct_change(3)
)

df["MA50_slope"] = (
    df["MA50"]
    .pct_change(3)
)


# ============================================================
# 6. ローソク足特徴
# ============================================================

df["body"] = (
    abs(
        df["Close"]
        - df["Open"]
    )
    / df["Open"]
)


df["range"] = (
    df["High"]
    - df["Low"]
) / df["Close"]


df["upper_wick"] = (

    df["High"]

    - df[
        [
            "Open",
            "Close"
        ]
    ].max(
        axis=1
    )

) / df["Close"]


df["lower_wick"] = (

    df[
        [
            "Open",
            "Close"
        ]
    ].min(
        axis=1
    )

    - df["Low"]

) / df["Close"]


df["bullish"] = (
    df["Close"]
    > df["Open"]
).astype(int)


# ============================================================
# 7. ボラティリティ
# ============================================================

df["volatility_1h"] = (
    df["return_5m"]
    .rolling(12)
    .std()
)

df["volatility_2h"] = (
    df["return_5m"]
    .rolling(24)
    .std()
)

df["volatility_4h"] = (
    df["return_5m"]
    .rolling(48)
    .std()
)


# ============================================================
# 8. RSI
# ============================================================

delta = (
    df["Close"]
    .diff()
)

gain = (
    delta
    .clip(lower=0)
)

loss = (
    -delta
    .clip(upper=0)
)

avg_gain = (
    gain
    .rolling(14)
    .mean()
)

avg_loss = (
    loss
    .rolling(14)
    .mean()
)

rs = (
    avg_gain
    / avg_loss
)

df["RSI"] = (
    100
    - 100
    / (
        1 + rs
    )
)


# ============================================================
# 9. 高値・安値距離
# ============================================================

df["high_1h"] = (
    df["High"]
    .rolling(12)
    .max()
)

df["low_1h"] = (
    df["Low"]
    .rolling(12)
    .min()
)


df["distance_high_1h"] = (
    df["Close"]
    / df["high_1h"]
    - 1
)

df["distance_low_1h"] = (
    df["Close"]
    / df["low_1h"]
    - 1
)


# ============================================================
# 10. 時刻情報
# ============================================================

df["hour"] = (
    df.index.hour
)

df["weekday"] = (
    df.index.dayofweek
)


df["hour_sin"] = np.sin(
    2
    * np.pi
    * df["hour"]
    / 24
)

df["hour_cos"] = np.cos(
    2
    * np.pi
    * df["hour"]
    / 24
)


# ============================================================
# 11. 特徴量
# ============================================================

move_features = [

    "volatility_1h",
    "volatility_2h",
    "volatility_4h",

    "range",
    "body",

    "return_5m",
    "return_15m",
    "return_30m",

    "MA20_slope",
    "MA50_slope",

    "distance_high_1h",
    "distance_low_1h",

    "hour_sin",
    "hour_cos",

    "weekday"
]


direction_features = [

    "return_5m",
    "return_15m",
    "return_30m",
    "return_1h",
    "return_2h",

    "MA5_distance",
    "MA20_distance",
    "MA50_distance",

    "MA5_slope",
    "MA20_slope",
    "MA50_slope",

    "RSI",

    "bullish",

    "body",
    "upper_wick",
    "lower_wick",

    "distance_high_1h",
    "distance_low_1h",

    "volatility_1h",

    "hour_sin",
    "hour_cos",

    "weekday"
]


all_features = list(
    set(
        move_features
        + direction_features
    )
)


# ============================================================
# 12. 時間減衰ウェイト
# ============================================================

def make_time_weights(
    index,
    half_life_days=20
):

    latest = (
        index.max()
    )

    age_days = (

        latest
        - index

    ).total_seconds() / 86400


    weights = (

        0.5
        ** (
            age_days
            / half_life_days
        )

    )


    return np.array(
        weights
    )


# ============================================================
# 13. Horizonごとの教師データ作成
# ============================================================

def make_horizon_data(
    horizon_bars
):

    temp = (
        df.copy()
    )


    # --------------------------------------------------------
    # 時間定義を完全統一
    #
    # シグナル時刻 = t
    #
    # entry
    # = t+1足のOpen
    #
    # 30分なら6本保有
    # = t+1 ～ t+6
    #
    # exit
    # = t+6足のClose
    # --------------------------------------------------------

    temp["entry_price"] = (
        temp["Open"]
        .shift(-1)
    )


    temp["exit_price"] = (
        temp["Close"]
        .shift(
            -horizon_bars
        )
    )


    temp["future_return"] = (

        temp[
            "exit_price"
        ]

        / temp[
            "entry_price"
        ]

        - 1
    )


    # MOVE
    temp["move_target"] = (

        abs(
            temp[
                "future_return"
            ]
        )

        > MOVE_THRESHOLD

    ).astype(int)


    # Direction
    temp[
        "direction_target"
    ] = np.where(

        temp[
            "future_return"
        ] > 0,

        1,

        0
    )


    columns = (

        all_features

        + [

            "entry_price",
            "exit_price",
            "future_return",

            "move_target",
            "direction_target"

        ]
    )


    data = (

        temp[
            columns
        ]

        .dropna()

        .copy()
    )


    return data


# ============================================================
# 14. TP / SLシミュレーション
# ============================================================

def simulate_trade(
    signal_time,
    direction,
    horizon_bars,
    tp,
    sl
):


    try:

        signal_loc = (
            df.index
            .get_loc(
                signal_time
            )
        )

    except KeyError:

        return np.nan, np.nan, np.nan


    # 次足Openでエントリー
    entry_loc = (
        signal_loc + 1
    )


    # 例:
    # horizon=6なら
    # t+1 ～ t+6
    exit_loc = (
        signal_loc
        + horizon_bars
    )


    if exit_loc >= len(df):

        return np.nan, np.nan, np.nan


    entry_price = (
        df.iloc[
            entry_loc
        ]["Open"]
    )


    # BUY
    if direction == "BUY":

        tp_price = (
            entry_price
            * (
                1 + tp
            )
        )

        sl_price = (
            entry_price
            * (
                1 - sl
            )
        )


    # SELL
    else:

        tp_price = (
            entry_price
            * (
                1 - tp
            )
        )

        sl_price = (
            entry_price
            * (
                1 + sl
            )
        )


    # MFE / MAE計算用
    window = (
        df.iloc[
            entry_loc:
            exit_loc + 1
        ]
    )


    if direction == "BUY":

        mfe = (
            window[
                "High"
            ].max()
            / entry_price
            - 1
        )

        mae = (
            window[
                "Low"
            ].min()
            / entry_price
            - 1
        )

    else:

        mfe = (
            entry_price
            / window[
                "Low"
            ].min()
            - 1
        )

        mae = (
            entry_price
            / window[
                "High"
            ].max()
            - 1
        )


    # --------------------------------------------------------
    # TP / SLの順番を調べる
    # --------------------------------------------------------

    for loc in range(
        entry_loc,
        exit_loc + 1
    ):


        high = (
            df.iloc[
                loc
            ]["High"]
        )

        low = (
            df.iloc[
                loc
            ]["Low"]
        )


        if direction == "BUY":

            tp_hit = (
                high >= tp_price
            )

            sl_hit = (
                low <= sl_price
            )

        else:

            tp_hit = (
                low <= tp_price
            )

            sl_hit = (
                high >= sl_price
            )


        # 同じ5分足内で両方なら
        # 保守的にSL扱い
        if tp_hit and sl_hit:

            return (
                -sl
                - TRADING_COST,
                mfe,
                mae
            )


        if tp_hit:

            return (
                tp
                - TRADING_COST,
                mfe,
                mae
            )


        if sl_hit:

            return (
                -sl
                - TRADING_COST,
                mfe,
                mae
            )


    # TP/SL未到達
    exit_price = (
        df.iloc[
            exit_loc
        ]["Close"]
    )


    if direction == "BUY":

        r = (
            exit_price
            / entry_price
            - 1
        )

    else:

        r = (
            entry_price
            / exit_price
            - 1
        )


    return (
        r
        - TRADING_COST,
        mfe,
        mae
    )


# ============================================================
# 15. Profit Factor
# ============================================================

def calculate_profit_factor(
    returns
):

    returns = np.array(
        returns
    )


    wins = (
        returns[
            returns > 0
        ]
    )

    losses = (
        returns[
            returns < 0
        ]
    )


    total_profit = (
        wins.sum()
        if len(wins) > 0
        else 0
    )


    total_loss = (
        abs(
            losses.sum()
        )
        if len(losses) > 0
        else 0
    )


    if total_loss == 0:

        return np.nan


    return (
        total_profit
        / total_loss
    )


print(
    "\n前半完了"
)

print(
    "次に後半セルを実行してください"
)

# ============================================================
# FX機械学習 完全修正版 後半
# ============================================================


# ============================================================
# 16. HorizonモデルをWalk-Forwardする関数
# ============================================================

def run_horizon_model(
    horizon_bars,
    horizon_name
):


    print(
        "\n\n########################################"
    )

    print(
        horizon_name,
        "MODEL"
    )

    print(
        "########################################"
    )


    data = (
        make_horizon_data(
            horizon_bars
        )
    )


    # GAPもHorizonに合わせる
    gap = (
        horizon_bars
    )


    block_size = (

        len(data)

        // (
            N_SPLITS + 1
        )
    )


    fold_results = []

    all_trades = []


    # ========================================================
    # Walk-Forward
    # ========================================================

    for fold in range(
        N_SPLITS
    ):


        train_end = (

            block_size
            * (
                fold + 1
            )
        )


        test_start = (

            train_end
            + gap
        )


        test_end = (

            test_start
            + block_size
        )


        if test_end > len(data):

            test_end = len(data)


        train_full = (

            data.iloc[
                :train_end
            ]
        )


        test = (

            data.iloc[
                test_start:test_end
            ]
        )


        if (
            len(train_full) < 500
            or
            len(test) == 0
        ):

            continue


        # ====================================================
        # 内部Validation
        # ====================================================

        inner_split = int(

            len(train_full)
            * 0.80

        )


        # Validationとの間にもGapを入れる
        train_core_end = (

            inner_split
            - gap
        )


        train_core = (

            train_full.iloc[
                :train_core_end
            ]
        )


        validation = (

            train_full.iloc[
                inner_split:
            ]
        )


        if (
            len(train_core) < 300
            or
            len(validation) < 100
        ):

            continue


        # ====================================================
        # MOVEモデル
        # ====================================================

        move_weights = (
            make_time_weights(
                train_core.index,
                HALF_LIFE_DAYS
            )
        )


        move_model = (
            RandomForestClassifier(

                n_estimators=350,

                max_depth=8,

                min_samples_leaf=20,

                max_features="sqrt",

                class_weight="balanced",

                random_state=42,

                n_jobs=-1
            )
        )


        move_model.fit(

            train_core[
                move_features
            ],

            train_core[
                "move_target"
            ],

            sample_weight=
                move_weights
        )


        # ====================================================
        # Directionモデル
        # ====================================================

        direction_core = (

            train_core[

                train_core[
                    "move_target"
                ] == 1

            ]
        )


        if len(
            direction_core
        ) < 100:

            continue


        direction_weights = (
            make_time_weights(

                direction_core.index,

                HALF_LIFE_DAYS
            )
        )


        direction_model = (
            RandomForestClassifier(

                n_estimators=350,

                max_depth=8,

                min_samples_leaf=15,

                max_features="sqrt",

                class_weight="balanced",

                random_state=42,

                n_jobs=-1
            )
        )


        direction_model.fit(

            direction_core[
                direction_features
            ],

            direction_core[
                "direction_target"
            ],

            sample_weight=
                direction_weights
        )


        # ====================================================
        # Validation確率
        # ====================================================

        val_p_move = (
            move_model
            .predict_proba(

                validation[
                    move_features
                ]

            )[:, 1]
        )


        val_direction_prob = (
            direction_model
            .predict_proba(

                validation[
                    direction_features
                ]

            )
        )


        class_map = {

            c: i

            for i, c

            in enumerate(
                direction_model.classes_
            )
        }


        val_p_down = (
            val_direction_prob[
                :,
                class_map[0]
            ]
        )


        val_p_up = (
            val_direction_prob[
                :,
                class_map[1]
            ]
        )


        # ====================================================
        # Validation内だけでボラ範囲決定
        # ====================================================

        vol_low = (
            train_core[
                "volatility_1h"
            ]
            .quantile(
                0.20
            )
        )


        vol_high = (
            train_core[
                "volatility_1h"
            ]
            .quantile(
                0.80
            )
        )


        # ====================================================
        # Validationだけで
        # Threshold + TP + SLを決める
        # ====================================================

        best_score = -999

        best_settings = None


        for move_t in MOVE_PROB_LIST:

            for direction_t in DIRECTION_PROB_LIST:

                for tp in TP_LIST:

                    for sl in SL_LIST:


                        signals = np.zeros(
                            len(validation)
                        )


                        valid_vol = (

                            (
                                validation[
                                    "volatility_1h"
                                ].values
                                >= vol_low
                            )

                            &

                            (
                                validation[
                                    "volatility_1h"
                                ].values
                                <= vol_high
                            )
                        )


                        buy_mask = (

                            valid_vol

                            &

                            (
                                val_p_move
                                >= move_t
                            )

                            &

                            (
                                val_p_up
                                >= direction_t
                            )

                            &

                            (
                                val_p_up
                                > val_p_down
                            )
                        )


                        sell_mask = (

                            valid_vol

                            &

                            (
                                val_p_move
                                >= move_t
                            )

                            &

                            (
                                val_p_down
                                >= direction_t
                            )

                            &

                            (
                                val_p_down
                                > val_p_up
                            )
                        )


                        signals[
                            buy_mask
                        ] = 1


                        signals[
                            sell_mask
                        ] = -1


                        validation_returns = []


                        i = 0


                        while i < len(
                            validation
                        ):


                            signal = (
                                signals[i]
                            )


                            if signal == 0:

                                i += 1
                                continue


                            time = (
                                validation.index[i]
                            )


                            if signal == 1:

                                direction = "BUY"

                            else:

                                direction = "SELL"


                            r, _, _ = (
                                simulate_trade(

                                    signal_time=
                                        time,

                                    direction=
                                        direction,

                                    horizon_bars=
                                        horizon_bars,

                                    tp=
                                        tp,

                                    sl=
                                        sl
                                )
                            )


                            if not np.isnan(r):

                                validation_returns.append(
                                    r
                                )


                            # ポジション保有中は
                            # 新規取引しない
                            i += (
                                horizon_bars
                            )


                        validation_returns = (
                            np.array(
                                validation_returns
                            )
                        )


                        # サンプル少なすぎる設定は無視
                        if len(
                            validation_returns
                        ) < 15:

                            continue


                        # 平均期待値を評価
                        score = (
                            validation_returns.mean()
                        )


                        if score > best_score:

                            best_score = score


                            best_settings = {

                                "move_threshold":
                                    move_t,

                                "direction_threshold":
                                    direction_t,

                                "tp":
                                    tp,

                                "sl":
                                    sl
                            }


        # 選べなければスキップ
        if best_settings is None:

            print(
                "Fold",
                fold + 1,
                "設定を選べず"
            )

            continue


        # ====================================================
        # 全Trainで再学習
        # ====================================================

        final_move_weights = (
            make_time_weights(

                train_full.index,

                HALF_LIFE_DAYS
            )
        )


        final_move_model = (
            RandomForestClassifier(

                n_estimators=500,

                max_depth=8,

                min_samples_leaf=20,

                max_features="sqrt",

                class_weight="balanced",

                random_state=42,

                n_jobs=-1
            )
        )


        final_move_model.fit(

            train_full[
                move_features
            ],

            train_full[
                "move_target"
            ],

            sample_weight=
                final_move_weights
        )


        final_direction_train = (

            train_full[

                train_full[
                    "move_target"
                ] == 1

            ]
        )


        final_direction_weights = (
            make_time_weights(

                final_direction_train.index,

                HALF_LIFE_DAYS
            )
        )


        final_direction_model = (
            RandomForestClassifier(

                n_estimators=500,

                max_depth=8,

                min_samples_leaf=15,

                max_features="sqrt",

                class_weight="balanced",

                random_state=42,

                n_jobs=-1
            )
        )


        final_direction_model.fit(

            final_direction_train[
                direction_features
            ],

            final_direction_train[
                "direction_target"
            ],

            sample_weight=
                final_direction_weights
        )


        # ====================================================
        # 完全未知Test予測
        # ====================================================

        test_p_move = (
            final_move_model
            .predict_proba(

                test[
                    move_features
                ]

            )[:, 1]
        )


        test_dir_prob = (
            final_direction_model
            .predict_proba(

                test[
                    direction_features
                ]

            )
        )


        test_class_map = {

            c: i

            for i, c

            in enumerate(
                final_direction_model.classes_
            )
        }


        test_p_down = (
            test_dir_prob[
                :,
                test_class_map[0]
            ]
        )


        test_p_up = (
            test_dir_prob[
                :,
                test_class_map[1]
            ]
        )


        move_t = (
            best_settings[
                "move_threshold"
            ]
        )


        direction_t = (
            best_settings[
                "direction_threshold"
            ]
        )


        tp = (
            best_settings[
                "tp"
            ]
        )


        sl = (
            best_settings[
                "sl"
            ]
        )


        valid_vol_test = (

            (
                test[
                    "volatility_1h"
                ].values
                >= vol_low
            )

            &

            (
                test[
                    "volatility_1h"
                ].values
                <= vol_high
            )
        )


        buy_test = (

            valid_vol_test

            &

            (
                test_p_move
                >= move_t
            )

            &

            (
                test_p_up
                >= direction_t
            )

            &

            (
                test_p_up
                > test_p_down
            )
        )


        sell_test = (

            valid_vol_test

            &

            (
                test_p_move
                >= move_t
            )

            &

            (
                test_p_down
                >= direction_t
            )

            &

            (
                test_p_down
                > test_p_up
            )
        )


        signals = np.zeros(
            len(test)
        )


        signals[
            buy_test
        ] = 1


        signals[
            sell_test
        ] = -1


        # ====================================================
        # Testバックテスト
        # ====================================================

        fold_returns = []

        fold_mfe = []

        fold_mae = []


        i = 0


        while i < len(
            test
        ):


            signal = (
                signals[i]
            )


            if signal == 0:

                i += 1
                continue


            time = (
                test.index[i]
            )


            if signal == 1:

                direction = "BUY"

            else:

                direction = "SELL"


            r, mfe, mae = (
                simulate_trade(

                    signal_time=
                        time,

                    direction=
                        direction,

                    horizon_bars=
                        horizon_bars,

                    tp=
                        tp,

                    sl=
                        sl
                )
            )


            if not np.isnan(r):

                fold_returns.append(
                    r
                )

                fold_mfe.append(
                    mfe
                )

                fold_mae.append(
                    mae
                )


                all_trades.append({

                    "model":
                        horizon_name,

                    "fold":
                        fold + 1,

                    "time":
                        time,

                    "direction":
                        direction,

                    "return":
                        r,

                    "MFE":
                        mfe,

                    "MAE":
                        mae,

                    "move_threshold":
                        move_t,

                    "direction_threshold":
                        direction_t,

                    "tp":
                        tp,

                    "sl":
                        sl
                })


            i += (
                horizon_bars
            )


        fold_returns = (
            np.array(
                fold_returns
            )
        )


        # ====================================================
        # Fold統計
        # ====================================================

        if len(
            fold_returns
        ) > 0:


            win_rate = (

                fold_returns
                > 0

            ).mean()


            avg_return = (
                fold_returns.mean()
            )


            profit_factor = (
                calculate_profit_factor(
                    fold_returns
                )
            )


            avg_mfe = (
                np.mean(
                    fold_mfe
                )
            )


            avg_mae = (
                np.mean(
                    fold_mae
                )
            )


        else:

            win_rate = np.nan

            avg_return = np.nan

            profit_factor = np.nan

            avg_mfe = np.nan

            avg_mae = np.nan


        fold_results.append({

            "fold":
                fold + 1,

            "trades":
                len(
                    fold_returns
                ),

            "win_rate":
                win_rate,

            "average_return":
                avg_return,

            "profit_factor":
                profit_factor,

            "avg_MFE":
                avg_mfe,

            "avg_MAE":
                avg_mae,

            "move_threshold":
                move_t,

            "direction_threshold":
                direction_t,

            "tp":
                tp,

            "sl":
                sl
        })


    # ========================================================
    # 結果
    # ========================================================

    fold_df = (
        pd.DataFrame(
            fold_results
        )
    )


    trades_df = (
        pd.DataFrame(
            all_trades
        )
    )


    if len(
        fold_df
    ) > 0:


        fold_df[
            "win_rate"
        ] *= 100

        fold_df[
            "average_return"
        ] *= 100

        fold_df[
            "avg_MFE"
        ] *= 100

        fold_df[
            "avg_MAE"
        ] *= 100

        fold_df[
            "tp"
        ] *= 100

        fold_df[
            "sl"
        ] *= 100


    print(
        "\n=============================="
    )

    print(
        horizon_name,
        "Fold結果"
    )

    print(
        "=============================="
    )

    print(
        fold_df
    )


    # ========================================================
    # 全取引統計
    # ========================================================

    if len(
        trades_df
    ) == 0:

        return (
            fold_df,
            trades_df
        )


    returns = (
        trades_df[
            "return"
        ].values
    )


    total_win_rate = (
        (
            returns > 0
        ).mean()
    )


    average_return = (
        returns.mean()
    )


    profit_factor = (
        calculate_profit_factor(
            returns
        )
    )


    print(
        "\n総取引数:",
        len(returns)
    )

    print(
        "全取引勝率:",
        round(
            total_win_rate
            * 100,
            2
        ),
        "%"
    )

    print(
        "期待値 / 取引:",
        round(
            average_return
            * 100,
            4
        ),
        "%"
    )

    print(
        "Profit Factor:",
        round(
            profit_factor,
            3
        )
    )


    # ========================================================
    # 最大DD
    # ========================================================

    equity = (
        1
        + pd.Series(
            returns
        )
    ).cumprod()


    running_max = (
        equity
        .cummax()
    )


    drawdown = (
        equity
        / running_max
        - 1
    )


    max_dd = (
        drawdown.min()
    )


    print(
        "最大DD:",
        round(
            max_dd
            * 100,
            3
        ),
        "%"
    )


    # ========================================================
    # 資金シミュレーション
    # ========================================================

    print(
        "\n1万円シミュレーション"
    )


    for leverage in LEVERAGES:


        capital = (
            INITIAL_CAPITAL
        )


        for r in returns:

            capital *= (

                1
                + r
                * leverage
            )


        profit = (
            capital
            - INITIAL_CAPITAL
        )


        print(

            leverage,
            "倍 →",

            round(
                capital,
                2
            ),

            "円",

            "利益:",
            round(
                profit,
                2
            ),
            "円"
        )


    # Equity
    plt.figure(
        figsize=(13, 5)
    )

    plt.plot(
        trades_df[
            "time"
        ],
        equity.values
    )

    plt.title(
        horizon_name
        + " Walk-Forward Equity"
    )

    plt.xlabel(
        "Time"
    )

    plt.ylabel(
        "Growth of 1"
    )

    plt.grid()

    plt.show()


    return (
        fold_df,
        trades_df
    )


# ============================================================
# 17. 30分モデル
# ============================================================

fold_30m, trades_30m = (
    run_horizon_model(

        horizon_bars=6,

        horizon_name="30 MIN"

    )
)


# ============================================================
# 18. 60分モデル
# ============================================================

fold_60m, trades_60m = (
    run_horizon_model(

        horizon_bars=12,

        horizon_name="60 MIN"

    )
)


# ============================================================
# 19. 30分 vs 60分 比較
# ============================================================

comparison = []


for name, trades_temp in [

    (
        "30 MIN",
        trades_30m
    ),

    (
        "60 MIN",
        trades_60m
    )

]:


    if len(
        trades_temp
    ) == 0:

        continue


    r = (
        trades_temp[
            "return"
        ].values
    )


    comparison.append({

        "model":
            name,

        "trades":
            len(r),

        "win_rate":
            (
                r > 0
            ).mean()
            * 100,

        "average_return":
            r.mean()
            * 100,

        "profit_factor":
            calculate_profit_factor(
                r
            ),

        "average_MFE":
            trades_temp[
                "MFE"
            ].mean()
            * 100,

        "average_MAE":
            trades_temp[
                "MAE"
            ].mean()
            * 100
    })


comparison_df = (
    pd.DataFrame(
        comparison
    )
)


print(
    "\n\n=============================="
)

print(
    "30分 vs 60分 最終比較"
)

print(
    "=============================="
)


print(
    comparison_df
)